## LSTM Model with Quarterly Financial Report Injection

This pipeline forecasts daily stock prices for Israeli companies using a stacked LSTM network. It integrates both technical indicators and quarterly financial metrics (extracted from TASE reports) to enhance prediction accuracy.

### Key Components

#### Data Sources
- Stock Price Data: OHLCV data (2020–2024) downloaded from Yahoo Finance.
- Financial Reports: Extracted and preprocessed from TASE's MAYA system. Parameters used: Net Income, Total Assets, Revenue.

#### Feature Engineering
- Technical Indicators:
  - High–Low and Open–Close differences
  - Rolling means: 5d, 10d, 20d
  - Volatility (10-day std), RSI, MACD, daily returns, up/down flag
- Financial Injection:
  - Precomputed correlations (metrics change, price change percentage, model score)
  - Injected only at official report dates (e.g., 31-Mar, 30-Jun)
  - No forward-filling: This allows the LSTM to learn the temporal influence of quarterly disclosures over time

#### Preprocessing
- Technical and financial features scaled separately using RobustScaler
- Financial features normalized only across non-zero entries (zero = padding)

---

### Model Architecture

| Layer               | Units | Notes                          |
|--------------------|-------|--------------------------------|
| LSTM × 4           | 100   | With Dropout (0.1)             |
| BatchNormalization | –     | After final LSTM layer         |
| Dense              | 25    | Followed by Dropout (0.1)      |
| Dense              | 1     | Final prediction output        |

- Optimizer: Adam
- Loss: Mean Squared Error (MSE)
- Epochs: 30  
- Batch Size: 16  
- Sequence Length: 1 day × concatenated features

---

### Evaluation and Forecasting

- Train/Test Split: Chronological 90/10 split
- Evaluation Metrics:
  - MSE, MAE, MAPE, R-squared
- Forecasting:
  - 30-day forward simulation based on normal-distributed percentage changes from model predictions

---

### Outputs

- Trained `.h5` model per ticker
- CSVs per ticker:
  - Actual vs. Predicted values (test set)
  - 30-day forecast
  - Evaluation metrics (`metrics_lstm.csv`)

This pipeline is designed to measure the added value of quarterly reports for sequential neural models and is directly comparable to equivalent pipelines using boosting-based models.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install lime
!pip install tensorflow

## Injection to main LSTM:

In [ ]:
# =========================
# 📦 Import necessary libraries
# =========================
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, BatchNormalization
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from datetime import datetime
import os
import warnings
warnings.filterwarnings("ignore")
os.sync()

# =========================
# 📚 Helper functions
# =========================
def calculate_rsi(prices, period=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calculate_macd(prices, fast=12, slow=26):
    exp1 = prices.ewm(span=fast, adjust=False).mean()
    exp2 = prices.ewm(span=slow, adjust=False).mean()
    return exp1 - exp2

def generate_report_dates(year):
    return [
        pd.Timestamp(f'{year}-03-31'),
        pd.Timestamp(f'{year}-06-30'),
        pd.Timestamp(f'{year}-09-30'),
        pd.Timestamp(f'{year}-12-31')
    ]

# =========================
# 📥 Load tickers
# =========================
ticker_df = pd.read_csv("/content/drive/Shareddrives/capstone project-stock market robo-advisor/symbols_Israel.csv")
tickers = ticker_df['0'].tolist()

# =========================
# 📈 Initialize result DataFrames
# =========================
actual_vs_pred_df = pd.DataFrame(columns=['Date', 'Ticker', 'Actual', 'Predicted'])
forecast_df = pd.DataFrame(columns=['Date', 'Ticker', 'Forecast', 'Days_Ahead'])
metrics_df = pd.DataFrame(columns=['Ticker', 'MSE', 'MAE', 'MAPE', 'R2'])

# =========================
# 🔁 Loop over each ticker
# =========================
start_date = '2020-01-01'
end_date = '2024-12-31'

for ticker in tickers:
    try:
        print(f"\n================ Processing {ticker} ================")

        stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
        if stock_data.empty:
            print(f"No data for {ticker}. Skipping.")
            continue

        stock_data.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
        stock_data['Adj Close'] = stock_data['Close']

        # Feature engineering
        stock_data['High_Low_Diff'] = stock_data['High'] - stock_data['Low']
        stock_data['Open_Close_Diff'] = stock_data['Open'] - stock_data['Close']
        stock_data['Adj_Close_5d_Rolling'] = stock_data['Adj Close'].rolling(window=5).mean()
        stock_data['Adj_Close_10d_Rolling'] = stock_data['Adj Close'].rolling(window=10).mean()
        stock_data['Adj_Close_20d_Rolling'] = stock_data['Adj Close'].rolling(window=20).mean()
        stock_data['Volatility'] = stock_data['Adj Close'].rolling(window=10).std()
        stock_data['Adj_Close_Up'] = (stock_data['Adj Close'].diff() > 0).astype(int)
        stock_data['RSI'] = calculate_rsi(stock_data['Adj Close'])
        stock_data['MACD'] = calculate_macd(stock_data['Adj Close'])
        stock_data['Returns'] = stock_data['Adj Close'].pct_change()

        # Financial injections
        metrics_features = ['Net Income', 'Total Assets', 'Revenue']
        financial_dates = []
        for y in range(2020, 2025):
            financial_dates.extend(generate_report_dates(y))
        financial_dates = pd.to_datetime(financial_dates)

        financial_features = pd.DataFrame(0.0, index=stock_data.index, columns=[
            f'{feature}_metrics_change' for feature in metrics_features] +
            [f'{feature}_price_change_last_day' for feature in metrics_features] +
            [f'{feature}_price_change_30d' for feature in metrics_features] +
            [f'{feature}_price_change_90d' for feature in metrics_features] +
            [f'{feature}_score_last_day' for feature in metrics_features] +
            [f'{feature}_score_30d' for feature in metrics_features] +
            [f'{feature}_score_90d' for feature in metrics_features]
        )

        for date in financial_dates:
            try:
                if date.year < 2020 or date.year > 2024:
                    continue
                year = date.year if date.month != 3 or date.year == 2020 else date.year - 1
                quarter = ('Q1' if date.month == 3 else 'Q2' if date.month == 6 else 'Q3' if date.month == 9 else 'Q4')
                next_quarter = ('Q2' if quarter == 'Q1' else 'Q3' if quarter == 'Q2' else 'Q4' if quarter == 'Q3' else 'Q1')
                next_year = year if quarter != 'Q4' else year + 1
                if next_year > 2024:
                    continue

                base_path = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/Correlations"
                metrics_path = f"{base_path}/correlation metrics change/{quarter}_{year}_to_{next_quarter}_{next_year}.csv"
                price_change_path = f"{base_path}/correlations price change/{quarter}_{year}_to_{next_quarter}_{next_year}.csv"
                score_path = f"{base_path}/correlation score/{quarter}_{year}_to_{next_quarter}_{next_year}.csv"

                correlation_metrics_change = pd.read_csv(metrics_path, index_col=0)
                correlation_price_change = pd.read_csv(price_change_path, index_col=0)
                correlation_score = pd.read_csv(score_path, index_col=0)

                if date not in financial_features.index:
                    continue

                for feature in metrics_features:
                    if ticker in correlation_metrics_change.index:
                        val = correlation_metrics_change.loc[ticker, feature]
                        val_list_price = eval(correlation_price_change.loc[ticker, feature])
                        val_list_score = eval(correlation_score.loc[ticker, feature])
                    else:
                        val = 0
                        val_list_price = [0, 0, 0]
                        val_list_score = [0, 0, 0]

                    financial_features.at[date, f'{feature}_metrics_change'] = val
                    financial_features.at[date, f'{feature}_price_change_last_day'] = val_list_price[0]
                    financial_features.at[date, f'{feature}_price_change_30d'] = val_list_price[1]
                    financial_features.at[date, f'{feature}_price_change_90d'] = val_list_price[2]
                    financial_features.at[date, f'{feature}_score_last_day'] = val_list_score[0]
                    financial_features.at[date, f'{feature}_score_30d'] = val_list_score[1]
                    financial_features.at[date, f'{feature}_score_90d'] = val_list_score[2]

            except Exception as e:
                print(f"Skipping financials for {date.date()}: {e}")

        full_data = pd.concat([stock_data, financial_features], axis=1)
        full_data = full_data.dropna()

        # Train/Test split
        X = full_data.drop(columns=['Adj Close'])
        y = full_data['Adj Close']

        price_features = ['Close', 'Open', 'High', 'Low', 'High_Low_Diff', 'Open_Close_Diff',
                          'Adj_Close_5d_Rolling', 'Adj_Close_10d_Rolling', 'Adj_Close_20d_Rolling',
                          'Volatility', 'Adj_Close_Up', 'RSI', 'MACD', 'Returns']
        financial_features_list = [col for col in X.columns if col not in price_features]

        scaler_price = RobustScaler()
        X_price_scaled = scaler_price.fit_transform(X[price_features])

        X_financial = X[financial_features_list].copy()
        X_financial_scaled = X_financial.copy()
        for col in financial_features_list:
            real_values = X_financial[col] != 0
            if real_values.sum() > 0:
                min_val = X_financial.loc[real_values, col].min()
                max_val = X_financial.loc[real_values, col].max()
                if max_val - min_val != 0:
                    X_financial_scaled.loc[real_values, col] = (X_financial.loc[real_values, col] - min_val) / (max_val - min_val)
                else:
                    X_financial_scaled.loc[real_values, col] = 0.5

        X_scaled = np.concatenate([X_price_scaled, X_financial_scaled.values], axis=1)
        X_scaled = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])

        scaler_y = RobustScaler()
        y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))

        split_idx = int(0.9 * len(X_scaled))
        X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
        y_train, y_test = y_scaled[:split_idx], y_scaled[split_idx:]
        train_dates, test_dates = full_data.index[:split_idx], full_data.index[split_idx:]

        # Model definition
        model = Sequential([
            Input(shape=(X_train.shape[1], X_train.shape[2])),
            LSTM(100, return_sequences=True),
            Dropout(0.1),
            LSTM(100, return_sequences=True),
            Dropout(0.1),
            LSTM(100, return_sequences=True),
            Dropout(0.1),
            LSTM(100),
            BatchNormalization(),
            Dropout(0.1),
            Dense(25),
            Dropout(0.1),
            Dense(1)
        ])

        model.compile(optimizer='adam', loss='mse')
        model.fit(X_train, y_train, epochs=30, batch_size=16, verbose=0)

        # Save model
        model_save_path = f"/content/drive/Shareddrives/capstone project-stock market robo-advisor/LSTM models with report parameters/{ticker}_lstm_model.h5"
        model.save(model_save_path)

        # Evaluation
        y_pred = model.predict(X_test)

        y_test_inv = scaler_y.inverse_transform(y_test)
        y_pred_inv = scaler_y.inverse_transform(y_pred)

        mse = mean_squared_error(y_test_inv, y_pred_inv)
        mae = mean_absolute_error(y_test_inv, y_pred_inv)
        mape = mean_absolute_percentage_error(y_test_inv, y_pred_inv)
        r2 = r2_score(y_test_inv, y_pred_inv)

        metrics_df = pd.concat([metrics_df, pd.DataFrame({
            'Ticker': [ticker],
            'MSE': [mse],
            'MAE': [mae],
            'MAPE': [mape],
            'R2': [r2]
        })], ignore_index=True)

        temp_actual_pred = pd.DataFrame({
            'Date': test_dates,
            'Ticker': ticker,
            'Actual': y_test_inv.flatten(),
            'Predicted': y_pred_inv.flatten()
        })

        actual_vs_pred_df = pd.concat([actual_vs_pred_df, temp_actual_pred], ignore_index=True)

        # Forecast
        num_days_to_predict = 30
        pct_changes = pd.Series(y_pred_inv.flatten()).pct_change().dropna()
        mean_pct_change = pct_changes.mean()
        std_pct_change = pct_changes.std()

        future_predictions = [y_pred_inv[-1]]
        for _ in range(num_days_to_predict):
            sampled_pct_change = np.random.normal(loc=mean_pct_change, scale=std_pct_change)
            new_forecast = future_predictions[-1] * (1 + sampled_pct_change)
            future_predictions.append(new_forecast)

        future_dates = pd.date_range(test_dates[-1] + pd.Timedelta(days=1), periods=num_days_to_predict)
        temp_forecast = pd.DataFrame({
            'Date': future_dates,
            'Ticker': ticker,
            'Forecast': future_predictions[1:],
            'Days_Ahead': np.arange(1, num_days_to_predict + 1)
        })

        forecast_df = pd.concat([forecast_df, temp_forecast], ignore_index=True)

        # Save updated CSVs after each ticker
        save_base = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/LSTM models with report parameters"
        actual_vs_pred_df.to_csv(os.path.join(save_base, "actual_vs_pred_lstm.csv"), index=False)
        forecast_df.to_csv(os.path.join(save_base, "forecast_lstm.csv"), index=False)
        metrics_df.to_csv(os.path.join(save_base, "metrics_lstm.csv"), index=False)
        !sync

    except Exception as e:
        print(f"Failed processing {ticker}: {e}")

print("\n✅ All tickers processed!")



================ Processing SPNTC.TA ================


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 222ms/step

================ Processing STG.TA ================


3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 475ms/step

================ Processing STRG.TA ================


1/4 ━━━━━━━━━━━━━━━━━━━━ 1s 600ms/step

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 226ms/step

================ Processing STRS.TA ================


...

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 218ms/step

================ Processing ZPRS.TA ================


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 835ms/step

================ Processing ZMH.TA ================


4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 228ms/step

================ Processing ZOOZ.TA ================


3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 329ms/step

================ Processing ZUR.TA ================


4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 324ms/step

✅ All tickers processed!
